# 2026_08_04 기준  26년 전체 KOSPI 200 옵션 데이터 전처리 파이프라인

In [9]:
from pathlib import Path

import pandas as pd


filename = "1st_processed_kospi_200_option_all_20260804.csv"
search_roots = (Path.cwd(), *Path.cwd().parents)
processed_path = next(
    (root / "data" / "processed" / filename for root in search_roots
     if (root / "data" / "processed" / filename).is_file()),
    None,
)
if processed_path is None:
    raise FileNotFoundError(f"Could not find data/processed/{filename}")

processed_options = pd.read_csv(processed_path)
processed_options.head()

,contract_code,underlying,option_type,contract_month,expiry_date,expiry_timestamp,expiry_source,expiry_is_assumed,strike,market_session,...,vendor_iv_raw,vendor_implied_volatility,settlement_price,volume,turnover_krw_million,open_interest,analysis_eligible,primary_smile_eligible,quality_flags,rejection_reasons
0,B0168625,KOSPI200,call,202608,2026-08-13,2026-08-13 06:20:00+00:00,DERIVED_SECOND_THURSDAY,True,625.0,day,...,64.0,0.64,NaN,2,173,106,True,True,EOD_CLOSE_NOT_MID|ASSUMED_QUOTE_TIMESTAMP,NaN
1,B0168700,KOSPI200,call,202608,2026-08-13,2026-08-13 06:20:00+00:00,DERIVED_SECOND_THURSDAY,True,700.0,day,...,96.0,0.96,NaN,1,69,4,True,True,EOD_CLOSE_NOT_MID|ASSUMED_QUOTE_TIMESTAMP,NaN
2,B0168740,KOSPI200,call,202608,2026-08-13,2026-08-13 06:20:00+00:00,DERIVED_SECOND_THURSDAY,True,740.0,day,...,96.0,0.96,NaN,5,293,5,True,True,EOD_CLOSE_NOT_MID|ASSUMED_QUOTE_TIMESTAMP,NaN
3,B0168837,KOSPI200,call,202608,2026-08-13,2026-08-13 06:20:00+00:00,DERIVED_SECOND_THURSDAY,True,837.5,day,...,92.0,0.92,NaN,2,68,2,True,True,EOD_CLOSE_NOT_MID|ASSUMED_QUOTE_TIMESTAMP,NaN
4,B0168860,KOSPI200,call,202608,2026-08-13,2026-08-13 06:20:00+00:00,DERIVED_SECOND_THURSDAY,True,860.0,night,...,NaN,NaN,154.5,5,152,50,True,False,EOD_CLOSE_NOT_MID|ASSUMED_QUOTE_TIMESTAMP,NaN


## raw, processed data 불러오기

In [8]:
from hashlib import sha256


raw_filename = "kospi_200_option_all_20260804.csv"
raw_path = next(
    (root / "data" / "raw" / raw_filename for root in search_roots
     if (root / "data" / "raw" / raw_filename).is_file()),
    None,
)
if raw_path is None:
    raise FileNotFoundError(f"Could not find data/raw/{raw_filename}")

raw_options = pd.read_csv(raw_path)
extraction_audit = pd.DataFrame(
    {
        "value": [
            str(raw_path),
            sha256(raw_path.read_bytes()).hexdigest(),
            len(raw_options), #number of raws
            len(processed_options),
            len(raw_options) - len(processed_options),
            list(processed_options.columns),
        ]
    },
    index=[
        "raw file",
        "raw SHA-256",
        "raw rows",
        "processed rows",
        "dropped rows",
        "processed columns",
    ],
)
extraction_audit

,value
raw file,/home/minseok/finance_projects/option-pricing-...
raw SHA-256,396916f8f8a972130618e9d6c538faf24985cbd377cabe...
raw rows,10448
processed rows,869
dropped rows,9579
processed columns,"[contract_code, underlying, option_type, contr..."


- 필수열들이 포함되는지 확인한다. ( 맨위에 필수 열 목록)
- 그리고 processed 의 한 열에 결측치 ( 빈칸 ) 이 하나라도 있는지 확인
- contract_code & quote_date & market_session 3개가 중복된 row가 있는지 검사하고, 개수를 센다.
- 가격, 행사가, 잔존만기 <= 0 인지 확인한다.
- 옵션유형이 call, put 중 하나인지 확인 / session이 day, night 중 하나인지 확인

In [10]:
required_columns = {
    "contract_code", "option_type", "strike", "market_session",
    "quote_date", "quote_timestamp", "T", "target_price",
}
core_columns = [
    "contract_code", "option_type", "strike", "market_session",
    "quote_date", "quote_timestamp", "T", "target_price",
]
missing_columns = sorted(required_columns - set(processed_options.columns))

if missing_columns:
    audit_failures = {
        "필수 열": len(missing_columns),
        "결측치": pd.NA,
        "중복 quote": pd.NA,
        "가격": pd.NA,
        "행사가": pd.NA,
        "잔존만기": pd.NA,
        "옵션 유형": pd.NA,
        "세션": pd.NA,
    }
else:
    audit_failures = {
        "필수 열": 0,
        "결측치": int(processed_options[core_columns].isna().any(axis=1).sum()), 
        "중복 quote": int(processed_options.duplicated(
            subset=["contract_code", "quote_date", "market_session"],
            keep=False,
        ).sum()),
        "가격": int(processed_options["target_price"].le(0).sum()),
        "행사가": int(processed_options["strike"].le(0).sum()),
        "잔존만기": int(processed_options["T"].le(0).sum()),
        "옵션 유형": int((~processed_options["option_type"].isin(["call", "put"])).sum()),
        "세션": int((~processed_options["market_session"].isin(["day", "night"])).sum()),
    }

processed_error_audit = pd.DataFrame.from_dict(
    audit_failures, orient="index", columns=["failed_rows"]
)
processed_error_audit["passed"] = processed_error_audit["failed_rows"].eq(0)
processed_error_audit.loc["필수 열", "details"] = (
    ", ".join(missing_columns) if missing_columns else "all required columns present"
)
processed_error_audit

,failed_rows,passed,details
필수 열,0,True,all required columns present
결측치,0,True,NaN
중복 quote,0,True,NaN
가격,0,True,NaN
행사가,0,True,NaN
잔존만기,0,True,NaN
옵션 유형,0,True,NaN
세션,0,True,NaN


In [7]:
september_2026_options = processed_options[
    (processed_options["market_session"] == "day")
    & (processed_options["contract_month"].astype("string") == "202609")
].copy()

september_2026_options

,contract_code,underlying,option_type,contract_month,expiry_date,expiry_timestamp,expiry_source,expiry_is_assumed,strike,market_session,...,vendor_iv_raw,vendor_implied_volatility,settlement_price,volume,turnover_krw_million,open_interest,analysis_eligible,primary_smile_eligible,quality_flags,rejection_reasons
430,B0169420,KOSPI200,call,202609,2026-09-10,2026-09-10 06:20:00+00:00,DERIVED_SECOND_THURSDAY,True,420.0,day,...,64.0,0.640,NaN,1,139,1,True,True,EOD_CLOSE_NOT_MID|ASSUMED_QUOTE_TIMESTAMP,NaN
431,B0169950,KOSPI200,call,202609,2026-09-10,2026-09-10 06:20:00+00:00,DERIVED_SECOND_THURSDAY,True,950.0,day,...,74.0,0.740,NaN,1,25,306,True,True,EOD_CLOSE_NOT_MID|ASSUMED_QUOTE_TIMESTAMP,NaN
432,B0169960,KOSPI200,call,202609,2026-09-10,2026-09-10 06:20:00+00:00,DERIVED_SECOND_THURSDAY,True,960.0,day,...,72.5,0.725,NaN,1,23,33,True,True,EOD_CLOSE_NOT_MID|ASSUMED_QUOTE_TIMESTAMP,NaN
433,B0169970,KOSPI200,call,202609,2026-09-10,2026-09-10 06:20:00+00:00,DERIVED_SECOND_THURSDAY,True,970.0,day,...,73.5,0.735,NaN,3,73,3,True,True,EOD_CLOSE_NOT_MID|ASSUMED_QUOTE_TIMESTAMP,NaN
434,B0169990,KOSPI200,call,202609,2026-09-10,2026-09-10 06:20:00+00:00,DERIVED_SECOND_THURSDAY,True,990.0,day,...,72.0,0.720,NaN,7,166,39,True,True,EOD_CLOSE_NOT_MID|ASSUMED_QUOTE_TIMESTAMP,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
850,C0169950,KOSPI200,put,202609,2026-09-10,2026-09-10 06:20:00+00:00,DERIVED_SECOND_THURSDAY,True,950.0,day,...,73.0,0.730,NaN,35,727,1217,True,True,EOD_CLOSE_NOT_MID|ASSUMED_QUOTE_TIMESTAMP,NaN
851,C0169960,KOSPI200,put,202609,2026-09-10,2026-09-10 06:20:00+00:00,DERIVED_SECOND_THURSDAY,True,960.0,day,...,70.5,0.705,NaN,3,56,380,True,True,EOD_CLOSE_NOT_MID|ASSUMED_QUOTE_TIMESTAMP,NaN
852,C0169970,KOSPI200,put,202609,2026-09-10,2026-09-10 06:20:00+00:00,DERIVED_SECOND_THURSDAY,True,970.0,day,...,72.5,0.725,NaN,7,145,7,True,True,EOD_CLOSE_NOT_MID|ASSUMED_QUOTE_TIMESTAMP,NaN
853,C0169A03,KOSPI200,put,202609,2026-09-10,2026-09-10 06:20:00+00:00,DERIVED_SECOND_THURSDAY,True,1010.0,day,...,73.0,0.730,NaN,5,133,434,True,True,EOD_CLOSE_NOT_MID|ASSUMED_QUOTE_TIMESTAMP,NaN


raw data "/data/raw/kospi_200_future_all_20260804.csv" 에서 확인 가능한  동일한 2026년 9월물 KOSPI200
선물의 종가는 1000이다.

In [10]:
kospi_future_price = 1000.0  # from kospi_200_future_all_20260804.csv , 202609 day 

september_2026_atm_options = september_2026_options[
    september_2026_options["strike"]
    .div(kospi_future_price)
    .between(0.85, 1.15, inclusive="both")
].copy()

september_2026_atm_options

,contract_code,underlying,option_type,contract_month,expiry_date,expiry_timestamp,expiry_source,expiry_is_assumed,strike,market_session,...,vendor_iv_raw,vendor_implied_volatility,settlement_price,volume,turnover_krw_million,open_interest,analysis_eligible,primary_smile_eligible,quality_flags,rejection_reasons
431,B0169950,KOSPI200,call,202609,2026-09-10,2026-09-10 06:20:00+00:00,DERIVED_SECOND_THURSDAY,True,950.0,day,...,74.0,0.740,NaN,1,25,306,True,True,EOD_CLOSE_NOT_MID|ASSUMED_QUOTE_TIMESTAMP,NaN
432,B0169960,KOSPI200,call,202609,2026-09-10,2026-09-10 06:20:00+00:00,DERIVED_SECOND_THURSDAY,True,960.0,day,...,72.5,0.725,NaN,1,23,33,True,True,EOD_CLOSE_NOT_MID|ASSUMED_QUOTE_TIMESTAMP,NaN
433,B0169970,KOSPI200,call,202609,2026-09-10,2026-09-10 06:20:00+00:00,DERIVED_SECOND_THURSDAY,True,970.0,day,...,73.5,0.735,NaN,3,73,3,True,True,EOD_CLOSE_NOT_MID|ASSUMED_QUOTE_TIMESTAMP,NaN
434,B0169990,KOSPI200,call,202609,2026-09-10,2026-09-10 06:20:00+00:00,DERIVED_SECOND_THURSDAY,True,990.0,day,...,72.0,0.720,NaN,7,166,39,True,True,EOD_CLOSE_NOT_MID|ASSUMED_QUOTE_TIMESTAMP,NaN
435,B0169A01,KOSPI200,call,202609,2026-09-10,2026-09-10 06:20:00+00:00,DERIVED_SECOND_THURSDAY,True,1000.0,day,...,76.0,0.760,NaN,801,15824,2975,True,True,EOD_CLOSE_NOT_MID|ASSUMED_QUOTE_TIMESTAMP,NaN
436,B0169A03,KOSPI200,call,202609,2026-09-10,2026-09-10 06:20:00+00:00,DERIVED_SECOND_THURSDAY,True,1010.0,day,...,71.5,0.715,NaN,8,171,21,True,True,EOD_CLOSE_NOT_MID|ASSUMED_QUOTE_TIMESTAMP,NaN
437,B0169A14,KOSPI200,call,202609,2026-09-10,2026-09-10 06:20:00+00:00,DERIVED_SECOND_THURSDAY,True,1040.0,day,...,72.5,0.725,NaN,10,160,36,True,True,EOD_CLOSE_NOT_MID|ASSUMED_QUOTE_TIMESTAMP,NaN
438,B0169A18,KOSPI200,call,202609,2026-09-10,2026-09-10 06:20:00+00:00,DERIVED_SECOND_THURSDAY,True,1060.0,day,...,72.0,0.720,NaN,3,44,8,True,True,EOD_CLOSE_NOT_MID|ASSUMED_QUOTE_TIMESTAMP,NaN
439,B0169A61,KOSPI200,call,202609,2026-09-10,2026-09-10 06:20:00+00:00,DERIVED_SECOND_THURSDAY,True,1150.0,day,...,69.5,0.695,NaN,133,1168,238,True,True,EOD_CLOSE_NOT_MID|ASSUMED_QUOTE_TIMESTAMP,NaN
845,C0169850,KOSPI200,put,202609,2026-09-10,2026-09-10 06:20:00+00:00,DERIVED_SECOND_THURSDAY,True,850.0,day,...,74.3,0.743,NaN,59,584,1828,True,True,EOD_CLOSE_NOT_MID|ASSUMED_QUOTE_TIMESTAMP,NaN
